Libraries and Data

In [2]:
import spacy
import pandas as pd
import re
from pathlib import Path

ROOT = Path('../../data/raw')
data = ROOT / 'cases.csv'
metadata = ROOT / 'metadata.csv'


Visualizing

In [3]:
data = pd.read_csv(data)
metadata = pd.read_csv(metadata)

data.head()
metadata.head()


,article_id,authors,case_amount,doi,journal,journal_detail,keywords,license,link,major_mesh_terms,mesh_terms,pmcid,pmid,title,year
0,PMC5137649,"[C E Bailey, M B Fritz, L Webb, N B Merchant, ...",1,10.1308/003588414X13824511649977,Ann R Coll Surg Engl,2014 Jan;96(1):88E-90E.,NaN,CC BY,https://pubmed.ncbi.nlm.nih.gov/24417851/,"[Cysts / diagnosis, Stomach / abnormalities, S...","[Cysts / diagnosis, Stomach / abnormalities, S...",PMC5137649,24417851,Gastric duplication cyst masquerading as a muc...,2014
1,PMC9387390,"[Elias A Chamely, Bryan Hoang, Nadim S Jafri, ...",1,10.4293/CRSLS.2021.00094,CRSLS,2022 Feb 25;9(1):e2021.00094.,"[delayed gastric emptying, endoscopy, gastric ...",CC BY-NC-SA,https://pubmed.ncbi.nlm.nih.gov/36016812/,"[Adenocarcinoma / complications, Gastric Bypas...","[Adenocarcinoma / complications, Gastric Bypas...",PMC9387390,36016812,Palliative Endoscopic Salvage of a Functionall...,2022
2,PMC3437073,[M Y Al-Naami],1,NaN,J Family Community Med,1999 Jan;6(1):45-8.,"[abscess, spleen, tuberculous]",CC BY-NC-SA,https://pubmed.ncbi.nlm.nih.gov/23008596/,[],[Case Reports],PMC3437073,23008596,An unusual presentation of tuberculous splenic...,1999
3,PMC7102447,"[Zeid Nesheiwat, Pinang Shastri, Rohit Vyas, C...",1,10.1155/2020/7842591,Case Rep Cardiol,2020 Jan 11;2020:7842591.,NaN,CC BY,https://pubmed.ncbi.nlm.nih.gov/32257451/,[],[Case Reports],PMC7102447,32257451,A Case of Acute Massive Bioprosthetic Mitral V...,2020
4,PMC6354154,"[Patrícia Alves, Inês Sá, Miguel Brito, Cátia ...",1,10.1155/2019/2537480,Case Rep Obstet Gynecol,2019 Jan 16;2019:2537480.,NaN,CC BY,https://pubmed.ncbi.nlm.nih.gov/30792930/,[],[Case Reports],PMC6354154,30792930,An Early Diagnosis of an Ovarian Steroid Cell ...,2019


inner joining important mesh terms

In [4]:
full_data = pd.merge(data, metadata)
full_data = full_data[['case_text','gender', 'case_id', 'major_mesh_terms']]
full_data.head()

,case_text,gender,case_id,major_mesh_terms
0,A 44-year-old woman presented with a 3-day his...,Female,PMC5137649_01,"[Cysts / diagnosis, Stomach / abnormalities, S..."
1,A 57-year-old man with no significant past med...,Male,PMC9387390_01,"[Adenocarcinoma / complications, Gastric Bypas..."
2,A 55-year-old male presented with a gradually ...,Male,PMC3437073_01,[]
3,A 65-year-old male with a past medical history...,Male,PMC7102447_01,[]
4,A 30-year-old nulligravida presented herself i...,Female,PMC6354154_01,[]


picking the first case_text

In [5]:
text = full_data['case_text'][0]
print(text)

A 44-year-old woman presented with a 3-day history of right flank and lower quadrant abdominal pain associated with nausea and constipation. Her past medical, family and medication history were otherwise non-contributory and her physical examination was unremarkable. She underwent contrast enhanced computed tomography, demonstrating a 6cm cystic lesion between the stomach and body/tail of the pancreas (Fig 1). She subsequently underwent EUS-FNA, which revealed normal pancreatic echotexture and a cyst measuring 6cm x 9cm that was free of internal septations or associated masses (Fig 2) but compressed the stomach. FNA of the cyst demonstrated no evidence of malignancy but did show the presence of extracellular mucin as well as a carcinoembryonic antigen (CEA) level of 12,476.5ng/ml and a carbohydrate antigen (CA) 19-9 level of 6iu/ml, suggesting the diagnosis of a mucinous pancreatic cystic neoplasm. The patient was therefore referred for surgical resection.   
A laparoscopic distal panc

setting dictionary

In [6]:
dicionario_medico = {
    "anatomy": [
        "stomach",
        "chest",
        "abdomen",
        "flank",
        "lung",
        "heart",
        "brain",
        "liver",
        "kidney",
        "skin",
        "head",
        "neck",
        "spine",
        "pelvis",
        "thoracic cavity",
        "left ventricle",
        "right atrium",
        "colorectal region",
        "bone marrow",
        "lymph nodes",
    ],
    "symptom": [
        "pain",
        "chest pain",
        "abdominal pain",
        "right flank pain",
        "fever",
        "nausea",
        "vomiting",
        "fatigue",
        "dizziness",
        "headache",
        "cough",
        "shortness of breath",
        "dyspnea",
        "diarrhea",
        "chills",
        "sweating",
        "palpitations",
        "weight loss",
        "edema",
        "swelling",
        "confusion",
        "weakness",
        "cyanosis",
    ],
    "diagnosis": [
        "myocardial infarction",
        "hypertension",
        "diabetes mellitus",
        "pneumonia",
        "stroke",
        "appendicitis",
        "atrial fibrillation",
        "asthma",
        "renal failure",
        "anemia",
        "chronic obstructive pulmonary disease",
        "pulmonary embolism",
        "sepsis",
        "gastroenteritis",
        "coronary artery disease",
        "cirrhosis",
    ],
    "treatment": [
        "surgery",
        "appendectomy",
        "chemotherapy",
        "radiation therapy",
        "dialysis",
        "intubation",
        "blood transfusion",
        "biopsy",
        "bypass surgery",
        "mechanical ventilation",
        "catheterization",
        "drainage",
    ],
    "medication": [
        "aspirin",
        "paracetamol",
        "ibuprofen",
        "insulin",
        "warfarin",
        "metformin",
        "amoxicillin",
        "omeprazole",
        "furosemide",
        "heparin",
        "atorvastatin",
        "enalapril",
        "morphine",
        "prednisone",
    ],
    "exam": [
        "blood test",
        "complete blood count",
        "cbc",
        "electrocardiogram",
        "ecg",
        "ekg",
        "chest X-ray",
        "computed tomography",
        "ct scan",
        "magnetic resonance imaging",
        "mri",
        "ultrasound",
        "urinalysis",
        "biopsy report",
        "echocardiogram",
        "blood glucose test",
    ],
}

In [8]:
nlp = spacy.load('en_core_web_sm')
processed_text = nlp(text)
without_stopwords_text = [token.text for token in processed_text if not token.is_stop and not token.is_punct and not '\n' in token.text]

print(without_stopwords_text)

['44', 'year', 'old', 'woman', 'presented', '3', 'day', 'history', 'right', 'flank', 'lower', 'quadrant', 'abdominal', 'pain', 'associated', 'nausea', 'constipation', 'past', 'medical', 'family', 'medication', 'history', 'non', 'contributory', 'physical', 'examination', 'unremarkable', 'underwent', 'contrast', 'enhanced', 'computed', 'tomography', 'demonstrating', '6', 'cm', 'cystic', 'lesion', 'stomach', 'body', 'tail', 'pancreas', 'Fig', '1', 'subsequently', 'underwent', 'EUS', 'FNA', 'revealed', 'normal', 'pancreatic', 'echotexture', 'cyst', 'measuring', '6', 'cm', 'x', '9', 'cm', 'free', 'internal', 'septations', 'associated', 'masses', 'Fig', '2', 'compressed', 'stomach', 'FNA', 'cyst', 'demonstrated', 'evidence', 'malignancy', 'presence', 'extracellular', 'mucin', 'carcinoembryonic', 'antigen', 'CEA', 'level', '12,476.5ng', 'ml', 'carbohydrate', 'antigen', '19', '9', 'level', '6iu', 'ml', 'suggesting', 'diagnosis', 'mucinous', 'pancreatic', 'cystic', 'neoplasm', 'patient', 'refer